# Daily run — June 20, 2026

Odds collected manually across five books for three World Cup matches. Helpers live in `odds_lib/`; this notebook is just the day's driver.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from odds_lib import process_match, upsert_csv

In [2]:
# 06/20/26 matches
new_odds = pd.DataFrame({
    "forecast_run_id": ["2026-06-20_morning"] * 45,

    "match_date": ["2026-06-20"] * 45,

    "match": (
        ["Netherlands vs Sweden"] * 15
        + ["Germany vs Ivory Coast"] * 15
        + ["Ecuador vs Curaçao"] * 15
    ),

    "question": ["Match result"] * 45,

    "outcome": (
        ["Netherlands", "Draw", "Sweden"] * 5
        + ["Germany", "Draw", "Ivory Coast"] * 5
        + ["Ecuador", "Draw", "Curaçao"] * 5
    ),

    "book": (
        ["Pinnacle"] * 3
        + ["Circa"] * 3
        + ["bet365"] * 3
        + ["DraftKings"] * 3
        + ["Caesars"] * 3
    ) * 3,

    "american_odds": [
        # Netherlands vs Sweden
        # Pinnacle: Netherlands, Draw, Sweden
        -138, 315, 390,
        # Circa: Netherlands, Draw, Sweden
        -140, 295, 388,
        # bet365: Netherlands, Draw, Sweden
        -143, 310, 350,
        # DraftKings: Netherlands, Draw, Sweden
        -138, 317, 376,
        # Caesars: Netherlands, Draw, Sweden
        -150, 290, 350,

        # Germany vs Ivory Coast
        # Pinnacle: Germany, Draw, Ivory Coast
        -195, 380, 560,
        # Circa: Germany, Draw, Ivory Coast
        -210, 380, 540,
        # bet365: Germany, Draw, Ivory Coast
        -209, 375, 500,
        # DraftKings: Germany, Draw, Ivory Coast
        -203, 376, 567,
        # Caesars: Germany, Draw, Ivory Coast
        -205, 340, 475,

        # Ecuador vs Curaçao
        # Pinnacle: Ecuador, Draw, Curaçao
        -660, 800, 2250,
        # Circa: Ecuador, Draw, Curaçao
        -800, 800, 2300,
        # bet365: Ecuador, Draw, Curaçao
        -800, 800, 1800,
        # DraftKings: Ecuador, Draw, Curaçao
        -669, 900, 1900,
        # Caesars: Ecuador, Draw, Curaçao
        -850, 700, 1900,
    ],

    "notes": [""] * 45,
})

In [3]:
df, consensus, submit_sums = process_match(new_odds)

display_cols = [
    "match",
    "outcome",
    "submit_prob",
    "mean_prob",
    "median_prob",
    "min_prob",
    "max_prob",
    "std_prob",
    "num_books",
]
consensus[display_cols]

,match,outcome,submit_prob,mean_prob,median_prob,min_prob,max_prob,std_prob,num_books
0,Netherlands vs Sweden,Netherlands,0.560651,0.560651,0.560135,0.556260,0.565757,0.003827,5
1,Netherlands vs Sweden,Draw,0.236018,0.236018,0.235115,0.231274,0.243096,0.004640,5
2,Netherlands vs Sweden,Sweden,0.203331,0.203331,0.204020,0.196769,0.210717,0.005545,5
3,Germany vs Ivory Coast,Germany,0.643258,0.643258,0.647506,0.626219,0.650469,0.010112,5
4,Germany vs Ivory Coast,Draw,0.203910,0.203910,0.203970,0.199822,0.211748,0.004847,5
5,Germany vs Ivory Coast,Ivory Coast,0.152831,0.152831,0.149952,0.145562,0.162033,0.006965,5
6,Ecuador vs Curaçao,Ecuador,0.847356,0.847356,0.849656,0.836408,0.853333,0.007080,5
7,Ecuador vs Curaçao,Draw,0.107165,0.107165,0.106667,0.098043,0.116851,0.006750,5
8,Ecuador vs Curaçao,Curaçao,0.045479,0.045479,0.046740,0.040000,0.050000,0.004455,5


## Log odds

Upsert keyed on `(forecast_run_id, match, question, book, outcome)` so re-running this cell replaces matching rows instead of duplicating them.

In [4]:
odds_to_save = new_odds.copy()
odds_to_save["entered_at"] = pd.Timestamp.now(tz="UTC")

upsert_csv(
    odds_to_save,
    "data/odds_log.csv",
    key_cols=["forecast_run_id", "match", "question", "book", "outcome"],
)